<a href="https://colab.research.google.com/github/soule-geophysics/geol-333-714/blob/main/notebooks/HW2b_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW2b: Reduction Density and the Depth to the Source

**Course:** GEOL 333 / 714, Geophysical Exploration Methods, Fall 2026
**Instructor:** Dax Soule (dax.soule@qc.cuny.edu)
**Due:** Sunday, October 11, 2026, 11:59 PM
**Submit:** This notebook (`.ipynb`) via Brightspace


## What you will do

This notebook continues the USF GeoPark sinkhole survey from HW2a. The reduction you carried out there is repeated below as given code, so you start from the same residual profile you handed in. Part numbering continues from HW2a: this notebook runs Parts 5 to 8. You will:

5. Test the reduction by asking whether the residual [anomaly](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#anomaly) still follows station elevation.
6. Sweep the slab density and let the data choose it.
7. Estimate the depth to the buried mass from the anomaly's half-width (Burger §6.7.1).
8. Reflect on how the choices made in the reduction decide where the sinkhole is.

## What you will hand in

This same notebook with the plots run and the short answers filled in. Save it as `HW2b_LASTNAME.ipynb` and upload to Brightspace.

## Data source and citation

> Parsekian, A. (n.d.). *IGUaNA Unit 3: Gravity and Magnetics Field Data Exercises, Part 1 (USF GeoPark sinkhole survey).* Science Education Resource Center, Carleton College. CC-BY-NC-SA 4.0. [SERC: IGUaNA Unit 3 teaching materials](https://serc.carleton.edu/iguana/teaching_materials/grav_mag/unit3.html)

The survey is a straight line across a campus field in Tampa, Florida, over karst limestone under a cover of sand and soil. The instrument was a relative gravimeter; readings are in milligals (mGal). The base station is at along-profile coordinate 100 m and was re-occupied between every science station. The survey ran in three field sessions:

| Date | Local time | Science stations read (along-profile m) |
|---|---|---|
| 2019-04-04 | 18:29 to 19:25 | 110, 120, 130, 140, 150 |
| 2019-04-06 | 10:27 to 13:18 | 50, 60, 70, 80, 90, 160, 170, 180, 190, 200 |
| 2019-04-18 | 15:52 to 17:12 | 0, 10, 20, 30, 40 |

The along-profile coordinate increases toward the south, so stations 0 to 90 are north of the base and stations 110 to 200 are south of it.

## Loading the data

The code below loads `profile.csv` automatically from a stable public web address, so in most cases you do not need to download anything: run the cells.

**If you have no internet, or the link is not live yet,** use the Brightspace fallback:

1. Download `profile.csv` from the Brightspace HW2b page.
2. In Google Colab, click the **📁 Files** icon in the left sidebar.
3. Drag `profile.csv` into the file panel.
4. In the loading cell below, comment out the `pd.read_csv(DATA_URL)` line and use the commented `pd.read_csv("profile.csv")` line instead.

> **Colab deletes uploaded files when the runtime disconnects.**
> If a CSV you uploaded by hand has vanished, re-run the data-loading cell
> (the URL load restores the data) or re-upload the file. Code and written
> answers persist in your own saved copy.

## Setup

These imports give us NumPy (numbers and arrays), Pandas (tables), and Plotly (interactive plots). All three come pre-installed in Colab; no `pip install` needed.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Course accessibility default: colorblind-safe qualitative palette
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe

## The reduction from HW2a, repeated

HW2a loaded the survey, fitted and removed the [drift](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#drift) on each of the three field days, tied the days to a common base reference, applied the free-air and slab corrections at the textbook density of 2.67 g/cm³, and fitted and removed a straight-line regional. It ended with a residual profile and a station holding its minimum.

The four cells below repeat that reduction, unchanged, so this notebook has the same columns to work with. They are given code with no questions attached: run them and check the printouts against the numbers you reported in HW2a. They should match.

What they leave behind, and what Parts 5 to 8 read: the columns `relative_gravity`, `fac_corrected`, `bouguer_2670`, `regional_2670` and `residual_2670` on the table `science`; `pooled_sd`, the survey's repeatability; `coeff_textbook`, the slab coefficient at 2.67 g/cm³; and `bouguer_coefficient`, the function that returns the slab attraction per metre at any density.

**Load.** Reads `profile.csv` into `df`, converts `time_est` to clock times and adds a `date` column so the drift fit can work one field day at a time. This is HW2a's Part 1.

In [ ]:
# Primary path: load directly from a stable public URL (no upload needed).
DATA_URL = "https://raw.githubusercontent.com/soule-geophysics/geol-333-714/main/data/profile.csv"
df = pd.read_csv(DATA_URL)

# Fallback (no internet, or URL not live yet): download profile.csv from the
# Brightspace HW2 page, drag it into the Colab file panel, then comment out the
# two lines above and use:
# df = pd.read_csv("profile.csv")

df['time_est'] = pd.to_datetime(df['time_est'])
df['date'] = df['time_est'].dt.date
df.head(10)

**Drift and base tie.** One field day at a time: a least-squares line through the day's base reads against time, subtracted from every reading on the day, then the day's mean corrected base read subtracted to put all three days on one reference. The last line prints `pooled_sd`, the scatter of all 23 corrected base reads about zero. That is the survey's noise floor, and Parts 7 and 8 measure the anomaly against it. This is HW2a's Part 2.

In [ ]:
df['relative_gravity'] = np.nan
corrected_base = []   # drift-corrected, base-tied base reads from every day

for d in sorted(df['date'].unique()):
    day_mask = df['date'] == d
    day = df[day_mask]
    day_base = day[day['is_base']]

    # Day-local time: 0 at the day's first reading.
    t0 = day['time_since_beg_min'].min()
    day_t = day['time_since_beg_min'] - t0
    base_t = day_base['time_since_beg_min'] - t0

    # 1. Fit the drift through the day's base reads. cov=True returns the covariance
    #    matrix; the square root of its [0, 0] entry is the 1-sigma uncertainty on
    #    the slope.
    (slope, intercept), cov = np.polyfit(base_t, day_base['gravity_mgal'], 1, cov=True)
    slope_sigma = np.sqrt(cov[0, 0])

    # 2. Drift-correct every reading on this day.
    drift_corrected = day['gravity_mgal'] - slope * day_t

    # 3. Base tie: subtract the day's mean drift-corrected base reading.
    base_reference = drift_corrected[day['is_base']].mean()
    df.loc[day_mask, 'relative_gravity'] = drift_corrected - base_reference

    base_after = drift_corrected[day['is_base']] - base_reference
    corrected_base.append(base_after.to_numpy())

    print(f'{d}: N_base = {len(day_base):2d}, span = {day_t.max():3.0f} min')
    print(f'    drift slope     = {slope:+.6f} +/- {slope_sigma:.6f} mGal/min (1 sigma)')
    print(f'    base reference  = {base_reference:.4f} mGal')
    print(f'    corrected base reads: SD = {base_after.std(ddof=1):.4f} mGal, '
          f'SE of the day mean = {base_after.std(ddof=1) / np.sqrt(len(day_base)):.4f} mGal')

# Pooled repeatability: one standard deviation over all 23 corrected base reads,
# with one degree of freedom removed for each day's mean.
all_base = np.concatenate(corrected_base)
n_days = len(corrected_base)
pooled_sd = np.sqrt(np.sum(all_base**2) / (len(all_base) - n_days))
print()
print(f'pooled SD of the {len(all_base)} drift-corrected base reads: {pooled_sd:.4f} mGal')
print('(the survey noise floor used in Parts 7 and 8)')

**Free air and the slab at 2.67 g/cm³.** Adds back 0.3086 mGal per metre of station elevation, then removes the slab attraction `2πGρh` at the textbook density, giving `bouguer_2670`. It also builds `science`, the twenty science stations sorted along the line. This is HW2a's Part 3.

In [ ]:
G = 6.674e-11                # m^3 / (kg s^2)
FAC_GRADIENT = 0.3086        # mGal per metre

def bouguer_coefficient(rho_kg_m3):
    """Slab attraction per metre of thickness, in mGal per metre."""
    return 2 * np.pi * G * rho_kg_m3 * 1e5    # 1 mGal = 1e-5 m/s^2

RHO_TEXTBOOK = 2670          # kg/m^3, which is 2.67 g/cm^3
coeff_textbook = bouguer_coefficient(RHO_TEXTBOOK)
print(f'Bouguer coefficient at 2.67 g/cm^3: {coeff_textbook:.4f} mGal per metre (SI route)')
print(f'board form 0.04193 x 2.67:          {0.04193 * 2.67:.4f} mGal per metre')

df['fac_corrected'] = df['relative_gravity'] + FAC_GRADIENT * df['elev_rel_base_m']
df['bouguer_2670'] = df['fac_corrected'] - coeff_textbook * df['elev_rel_base_m']

science = df[~df['is_base']].sort_values('point_along_profile_m').reset_index(drop=True)
table = science[['point_along_profile_m', 'date', 'elev_rel_base_m',
                 'relative_gravity', 'fac_corrected', 'bouguer_2670']]
print()
print(table.round(4).to_string(index=False))

h = science['elev_rel_base_m']
print()
print(f'largest free-air term on the line:   {(FAC_GRADIENT * h).abs().max():.4f} mGal')
print(f'largest slab term at 2.67 g/cm^3:    {(coeff_textbook * h).abs().max():.4f} mGal')

**The regional and the residual.** Fits a straight line to `bouguer_2670` against along-profile distance and subtracts it, leaving `residual_2670`. The printout gives the slope with its 1-sigma uncertainty and the station-by-station table. This is HW2a's Part 4.

In [ ]:
x = science['point_along_profile_m'].to_numpy()
anomaly_2670 = science['bouguer_2670'].to_numpy()

# Straight-line regional by least squares against along-profile distance. cov=True
# gives the covariance matrix; the square root of its [0, 0] entry is the 1-sigma
# uncertainty on the slope, as in the drift fit above.
(trend_slope, trend_intercept), trend_cov = np.polyfit(x, anomaly_2670, 1, cov=True)
trend_slope_sigma = np.sqrt(trend_cov[0, 0])

df['regional_2670'] = trend_slope * df['point_along_profile_m'] + trend_intercept
df['residual_2670'] = df['bouguer_2670'] - df['regional_2670']
science = df[~df['is_base']].sort_values('point_along_profile_m').reset_index(drop=True)

line_length = x.max() - x.min()                                   # metres
elev_trend = np.polyfit(x, science['elev_rel_base_m'], 1)[0]       # m of elevation per m along the line

print(f'regional slope:                  {trend_slope:+.6f} +/- {trend_slope_sigma:.6f} mGal per metre (1 sigma)')
print(f'change across the {line_length:.0f} m line:    {trend_slope * line_length:+.4f} mGal')
print(f'trend of station elevation:      {elev_trend:+.5f} m of elevation per metre along the line')
print()
print(science[['point_along_profile_m', 'elev_rel_base_m', 'bouguer_2670', 'regional_2670', 'residual_2670']]
      .round(4).to_string(index=False))

# The same step with a second-order polynomial as the regional, for comparison.
residual_quadratic = anomaly_2670 - np.polyval(np.polyfit(x, anomaly_2670, 2), x)
print()
for label, r in [('straight line', science['residual_2670'].to_numpy()),
                 ('second-order polynomial', residual_quadratic)]:
    i = int(np.argmin(r))
    print(f'regional = {label:24s} residual minimum at {x[i]:.0f} m, value {r[i]:+.4f} mGal')


## Part 5: The problem

Elevation is the quantity the two corrections in HW2a's Part 3 exist to remove. A reduced profile therefore has one test available before any interpretation: do the corrected values still depend on station elevation? The cell below plots the residual from HW2a's Part 4 against `elev_rel_base_m` and prints the [correlation coefficient](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#correlation) `r` between gravity and elevation at four stages of the reduction: the relative gravity before either elevation correction, after the free-air correction alone, after the slab, and after the slab and the regional. It also fits a line to the residual against elevation and prints its slope in mGal per metre.

This cell reads `elev_rel_base_m` from `science` as `h` and works through four columns in order: `relative_gravity`, `fac_corrected`, `bouguer_2670` and `residual_2670`. For each, `np.corrcoef` returns the correlation coefficient `r` between that column and `h`, which the cell prints with a label. It then fits a straight line of `residual_2670` against `h` with `np.polyfit`, stores the slope as `slope_vs_h` and the intercept as `intercept_vs_h`, and prints the slope in mGal per metre beside `coeff_textbook`, the slab coefficient HW2a's Part 3 removed. The next cell draws the fitted line.

In [ ]:
h = science['elev_rel_base_m']

for label, column in [('relative gravity (before elevation corrections)', 'relative_gravity'),
                      ('after free-air only', 'fac_corrected'),
                      ('after free-air and slab at 2.67 g/cm^3', 'bouguer_2670'),
                      ('after the slab and the regional (residual)', 'residual_2670')]:
    r = np.corrcoef(science[column], h)[0, 1]
    print(f'r(gravity, elevation) {label:48s} {r:+.3f}')

slope_vs_h, intercept_vs_h = np.polyfit(h, science['residual_2670'], 1)
print()
print(f'fitted slope of the residual against elevation: {slope_vs_h:+.4f} mGal per metre')
print(f'slab coefficient that was removed:              {coeff_textbook:.4f} mGal per metre')


This cell plots `residual_2670` against `elev_rel_base_m` from `science` with `px.scatter`, one marker per science station colored by field day, with `hover_data` set so that pointing at a marker shows its along-profile position. `np.linspace` makes fifty evenly spaced elevations between the lowest and highest station, and `fig.add_scatter` draws the line `slope_vs_h × h + intercept_vs_h` through them as a dashed line, with the slope in the legend.

In [ ]:
fig = px.scatter(
    science,
    x='elev_rel_base_m',
    y='residual_2670',
    color=science['date'].astype(str),
    hover_data=['point_along_profile_m'],
    title='Residual against station elevation, slab density 2.67 g/cm³',
    labels={'elev_rel_base_m': 'Elevation relative to base (m)',
            'residual_2670': 'Residual (mGal)', 'color': 'Date'},
)
fig.update_traces(marker=dict(size=12))
h_line = np.linspace(h.min(), h.max(), 50)
fig.add_scatter(
    x=h_line, y=slope_vs_h * h_line + intercept_vs_h, mode='lines',
    name=f'fit: {slope_vs_h:+.4f} mGal/m',
    line=dict(color=px.colors.qualitative.Safe[3], width=3, dash='dash'),
)
fig.show()

**Figure description:** A scatter plot of the residual (mGal, y-axis) against station elevation relative to the base (metres, x-axis), one marker per science station colored by field day, with the fitted straight line drawn dashed. Hovering a marker shows its along-profile position. Non-visual path: the previous cell prints the correlation at each stage and the slope of the fitted line.

**Question 5.1.** The printout gives the correlation between gravity and elevation at four stages.

1. The correlation changes sign across the first three stages. Explain the sign at each of those stages in terms of what the correction did to the readings.
2. What should the correlation be after a reduction that has removed the elevation dependence? Given the printed value after the slab and the regional, has this one?
3. The lowest residual in HW2a's Part 4 sits at the highest station on the line. How confident are you that the low at 180 m is geology?

*(Your answer):*

**Question 5.2.** The fitted slope of the residual against elevation is printed in mGal per metre, next to the slab coefficient that HW2a's Part 3 removed.

1. If the residual still falls with elevation after the slab was subtracted, did the slab remove too much per metre or too little?
2. Use the two printed numbers to estimate what the slab coefficient should have been, in mGal per metre, for the residual to have no dependence on elevation.
3. Convert that coefficient to a density in g/cm³ using the board form, 0.04193 mGal per metre for each g/cm³. Part 6 checks your number.

*(Your answer):*

## Part 6: The density sweep

Nettleton (1939) proposed letting the survey choose its own reduction density. Reduce the profile at a range of densities and keep the one at which the anomaly is least [correlated](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#correlation) with topography, on the reasoning that buried geology has no cause to follow the shape of the ground surface.

The cell below sweeps the slab density from 0 to 3000 kg/m³. At each density it reduces the profile, fits and removes a straight-line regional as in HW2a's Part 4, and computes the correlation between the residual and elevation. It plots the correlation against density and finds the density where the correlation crosses zero by linear interpolation. It then prints where the minimum of the residual sits at that density, alongside a short table of what other densities give. The second cell plots the two residual profiles, at 2.67 g/cm³ and at the density the sweep chose, on one set of axes.

> Nettleton, L. L. (1939). Determination of density for reduction of gravimeter observations. *Geophysics*, 4(3), 176–183.

This cell runs the density sweep described above and produces the printout and the first figure.

- It reads `elev_rel_base_m`, `point_along_profile_m` and `fac_corrected` from `science` into the arrays `h`, `x` and `fac`. Starting from the free-air-corrected values lets the slab term be recomputed at any density.
- `residual_at(rho_kg_m3)` subtracts the slab term at the given density, using `bouguer_coefficient` from HW2a's Part 3, then fits and removes a straight-line regional with `np.polyfit` and `np.polyval` as in HW2a's Part 4. It returns the residual at the twenty stations.
- `rho_sweep` is every density from 0 to 3000 kg/m³ in steps of 10. For each, `np.corrcoef` gives the correlation between `residual_at(rho)` and `h`; the results form the array `r_sweep`.
- `np.sign` and `np.diff` locate the first pair of neighbouring densities where `r_sweep` changes sign. Linear interpolation between them gives `rho_nettleton`, the density where the correlation crosses zero.
- It adds `residual_nettleton`, the residual at that density, as a new column on `science`.
- It prints `rho_nettleton` in kg/m³ and g/cm³, the correlation there and the slab coefficient there, then a table for ten densities giving the correlation, the station holding the residual minimum and the residual value there. `fmt_r` formats each correlation to three decimals with its sign.
- `px.line` plots `r_sweep` against density in g/cm³; `add_hline` marks zero correlation and two `add_vline` calls mark `rho_nettleton` and 2.67 g/cm³.

Part 7 reads `rho_nettleton` and `residual_nettleton`.

In [ ]:
h = science['elev_rel_base_m'].to_numpy()
x = science['point_along_profile_m'].to_numpy()
fac = science['fac_corrected'].to_numpy()

def residual_at(rho_kg_m3):
    """Bouguer anomaly of the science stations at one slab density, with a
    straight-line regional fitted and removed as in the reduction above."""
    anomaly = fac - bouguer_coefficient(rho_kg_m3) * h
    return anomaly - np.polyval(np.polyfit(x, anomaly, 1), x)

rho_sweep = np.arange(0, 3001, 10)                          # kg/m^3
r_sweep = np.array([np.corrcoef(residual_at(rho), h)[0, 1] for rho in rho_sweep])

# Zero crossing of r by linear interpolation between the two bracketing densities.
k = np.where(np.diff(np.sign(r_sweep)))[0][0]
rho_nettleton = rho_sweep[k] + (0 - r_sweep[k]) * (rho_sweep[k + 1] - rho_sweep[k]) / (r_sweep[k + 1] - r_sweep[k])
r_nettleton = np.corrcoef(residual_at(rho_nettleton), h)[0, 1]

def fmt_r(r):
    """Signed three-decimal correlation, with a clean zero at the crossing."""
    return f'{(0.0 if abs(r) < 0.0005 else r):+.3f}'

science['residual_nettleton'] = residual_at(rho_nettleton)

print(f'density where r crosses zero:  {rho_nettleton:.0f} kg/m^3  ({rho_nettleton / 1000:.3f} g/cm^3)')
print(f'correlation at that density:   {fmt_r(r_nettleton)}')
print(f'slab coefficient there:        {bouguer_coefficient(rho_nettleton):.4f} mGal per metre')
print()
print('density (g/cm^3)   r(residual, elevation)   station with the minimum   residual there (mGal)')
for rho in [0, 1000, 1200, rho_nettleton, 1600, 1800, 2000, 2300, 2670, 3000]:
    a = residual_at(rho)
    i = int(np.argmin(a))
    print(f'{rho / 1000:8.3f}           {fmt_r(np.corrcoef(a, h)[0, 1])}                    {x[i]:5.0f} m                {a[i]:+.4f}')

fig = px.line(
    x=rho_sweep / 1000, y=r_sweep,
    title='Correlation of the residual with elevation against slab density',
    labels={'x': 'Slab density (g/cm³)', 'y': 'r(residual, elevation)'},
)
fig.add_hline(y=0, line_dash='dash')
fig.add_vline(x=rho_nettleton / 1000, line_dash='dot',
              annotation_text=f'r = 0 at {rho_nettleton / 1000:.2f} g/cm³')
fig.add_vline(x=2.67, line_dash='dot', annotation_text='2.67 g/cm³')
fig.show()


**Figure description:** A line plot of the correlation between the residual and station elevation (y-axis, from +1 to −1) against slab density in g/cm³ (x-axis, 0 to 3). The line falls steadily from positive at zero density to negative at high density and crosses zero once; dotted vertical lines mark the crossing density and 2.67 g/cm³, and a dashed horizontal line marks zero correlation. Non-visual path: the previous cell prints the crossing density and a table of the correlation at ten densities.

This cell reshapes `science` with `melt` so that the two residual columns, `residual_2670` and `residual_nettleton`, stack into one column `residual_mgal`, with a second column `reduction` naming which reduction each value came from. The `map` call replaces the column names with legend labels that include the two densities. `px.scatter` then plots both residuals against `point_along_profile_m` on one set of axes, distinguished by color and marker shape. After the figure the cell prints position, elevation and both residuals for every station.

In [ ]:
plot_df = science.melt(
    id_vars=['point_along_profile_m', 'elev_rel_base_m'],
    value_vars=['residual_2670', 'residual_nettleton'],
    var_name='reduction', value_name='residual_mgal',
)
plot_df['reduction'] = plot_df['reduction'].map({
    'residual_2670': 'slab at 2.67 g/cm³, regional removed',
    'residual_nettleton': f'slab at {rho_nettleton / 1000:.2f} g/cm³ (sweep), regional removed',
})
fig = px.scatter(
    plot_df,
    x='point_along_profile_m', y='residual_mgal',
    color='reduction', symbol='reduction',
    title='Residual along the profile at two slab densities',
    labels={'point_along_profile_m': 'Along-profile distance (m); base at 100 m',
            'residual_mgal': 'Residual (mGal)', 'reduction': 'Reduction'},
)
fig.update_traces(marker=dict(size=12))
fig.show()

print(science[['point_along_profile_m', 'elev_rel_base_m', 'residual_2670', 'residual_nettleton']]
      .round(4).to_string(index=False))


**Figure description:** A scatter plot of the residual (mGal, y-axis) against along-profile distance (metres, x-axis) with two series distinguished by color and marker shape: the reduction at 2.67 g/cm³ and the reduction at the density the sweep chose, each with its straight-line regional removed. The two profiles have their lowest points at different stations. Non-visual path: the cell prints both residual columns for every station beside its elevation.

Typical bulk densities quoted in exploration texts, in g/cm³: dry sand and soil 1.4 to 1.8; water-saturated sand 1.9 to 2.2; limestone 1.9 to 2.9 depending on porosity, with cavernous karst at the low end; average continental crust 2.67.

**Question 6.1.** The printout reports a correlation of 0.000 at the density the sweep chose.

1. Is that zero a result of the survey, or a consequence of how the density was chosen? Explain.
2. Name two things in this part that are results, meaning the data could have come out otherwise.

*(Your answer):*

**Question 6.2.** Compare the two residual profiles.

1. Where is the minimum at the sweep density, what is its value, and where was it at 2.67 g/cm³?
2. From the sweep table, between which two densities does the station holding the minimum change?
3. Station 180, which held the minimum at 2.67 g/cm³, is the highest point on the line. Using the elevation column, where does station 110 sit? What does it tell you that changing one number in the reduction moves the sinkhole between those two positions?
4. Compare the density you estimated by hand in Question 5.2 with the sweep's value. The sweep refits the regional line at every density, and the reduction above printed the trend of station elevation along the line. Account for the difference between the two estimates.

*(Your answer):*

**Question 6.3.** The slab in this reduction is the material between each station's elevation and the base elevation, a layer at most 2.4 m thick at the surface of a sand-covered field.

1. Given the density ranges above, is the density the sweep chose geologically plausible for that layer? Which listed material does it match?
2. Was 2.67 g/cm³ ever the right density for this slab? What does 2.67 describe, and why does the textbook quote it?
3. In one sentence, what should a survey report state about its reduction density, and why?

*(Your answer):*

## Part 7: Estimate the depth to the buried mass

You have located where the anomaly is. Now estimate how deep its source is. This is the first time in the course you go from a measurement to an inferred subsurface property, which makes it a small inverse problem: working backward from the field you observe to the body that produced it.

Burger gives a shortcut. The simple-shape anomaly formulas are in §6.5, and the depth rule is the **half-maximum technique** of §6.7.1, which Week 4 derives at the board. For an anomaly whose source can be approximated by a simple body, the **half-width** $x_{1/2}$, the horizontal distance from the anomaly's peak out to where it has fallen to half its peak amplitude above the background, fixes the depth $z$ to the centre of the body:

$$\textbf{Sphere (compact body):}\quad z \approx 1.305\, x_{1/2} \qquad \text{(Eq. 6.54)}$$
$$\textbf{Horizontal cylinder (long body):}\quad z \approx x_{1/2} \qquad \text{(Eq. 6.55)}$$

A wider anomaly means a deeper source, and the same half-width implies a deeper centre for a sphere than for a cylinder.

Two choices are made in the cell below and both are named here. The **background** is the [median](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#median) of the twenty station residuals at the sweep density, a level the anomaly is measured from. The anomaly amplitude is the minimum minus that background. The **half level** is halfway between the background and the minimum, and the cell finds where the profile crosses it on each side of the minimum by linear interpolation between adjacent stations. The half-width is half the distance between the two crossings.

The cell also computes the one quantity a profile fixes on its own. Burger §6.5.2 shows that the anomaly of a sphere depends on its **excess mass**, the product of its volume and its density contrast, so the peak amplitude and the depth together give

$$M_e = \frac{\Delta g_\text{max}\, z^2}{G}$$

with $\Delta g_\text{max}$ in m/s². Finally it compares the anomaly amplitude with the pooled repeatability from HW2a's Part 2, at both slab densities, each with its regional removed.

This cell carries out the half-width construction described above on the residual at the sweep density.

- It reads `point_along_profile_m` and `residual_nettleton` from `science` into the arrays `x` and `a`.
- `np.median(a)` gives `background`. `np.argmin(a)` gives the index of the minimum, from which come `x_min` and `a_min`. `amplitude` is the minimum minus the background, and `half_level` is the background plus half the amplitude.
- `crossing(direction)` starts at the minimum and steps station by station in one direction along `a` until the profile passes through the half level, then interpolates linearly between the two bracketing stations to return the along-profile position of the crossing. It is called twice, toward smaller `x` and toward larger `x`, to give `x_left` and `x_right`.
- `half_width` is half the distance between the two crossings. `z_sphere` is 1.305 times it (Eq. 6.54) and `z_cylinder` equals it (Eq. 6.55).
- `excess_mass` applies the formula above, with the amplitude converted from mGal to m/s² and `z_sphere` as the depth.
- It prints the density used, the position and value of the minimum, the background, amplitude, half level, both crossings and the half-width, then the two depths and the excess mass.
- Finally it computes the amplitude of `residual_2670` in the same way (minimum minus median) and prints the ratio of amplitude to `pooled_sd` from HW2a's Part 2 at both densities, with the position of the minimum at 2.67 g/cm³.

The next cell draws the construction.

In [ ]:
x = science['point_along_profile_m'].to_numpy()
a = science['residual_nettleton'].to_numpy()

background = np.median(a)
i_min = int(np.argmin(a))
x_min, a_min = x[i_min], a[i_min]
amplitude = a_min - background                # negative for a mass deficit
half_level = background + amplitude / 2.0

def crossing(direction):
    """Along-profile position where the profile crosses the half level, walking
    from the minimum in the given direction (-1 toward smaller x, +1 toward larger)."""
    j = i_min
    while 0 <= j + direction < len(a):
        y1, y2 = a[j], a[j + direction]
        if min(y1, y2) <= half_level <= max(y1, y2):
            return x[j] + (half_level - y1) * (x[j + direction] - x[j]) / (y2 - y1)
        j += direction
    return np.nan

x_left, x_right = crossing(-1), crossing(+1)
half_width = (x_right - x_left) / 2.0
z_sphere = 1.305 * half_width
z_cylinder = half_width
excess_mass = abs(amplitude) * 1e-5 * z_sphere**2 / G     # kg, sphere depth

print(f'slab density used:          {rho_nettleton / 1000:.3f} g/cm^3')
print(f'residual minimum:           x = {x_min:.0f} m, residual = {a_min:+.4f} mGal')
print(f'background (median):        {background:+.4f} mGal')
print(f'amplitude below background: {amplitude:+.4f} mGal')
print(f'half level:                 {half_level:+.4f} mGal')
print(f'half-level crossings:       {x_left:.1f} m and {x_right:.1f} m')
print(f'half-width x_1/2:           {half_width:.2f} m')
print()
print(f'depth if sphere   (z = 1.305 x_1/2): {z_sphere:.1f} m')
print(f'depth if cylinder (z = x_1/2):       {z_cylinder:.1f} m')
print(f'excess mass, sphere at {z_sphere:.1f} m:     {excess_mass:.1e} kg')
print()
a_2670 = science['residual_2670'].to_numpy()
amp_2670 = a_2670.min() - np.median(a_2670)
print(f'amplitude / pooled base repeatability at {rho_nettleton / 1000:.2f} g/cm^3: '
      f'{abs(amplitude) / pooled_sd:.1f}  (amplitude {amplitude:+.4f}, noise {pooled_sd:.4f} mGal)')
print(f'amplitude / pooled base repeatability at 2.67 g/cm^3: '
      f'{abs(amp_2670) / pooled_sd:.1f}  (amplitude {amp_2670:+.4f}, minimum at {x[np.argmin(a_2670)]:.0f} m)')

This cell plots `residual_nettleton` against `point_along_profile_m` with `px.scatter` and joins the stations with a thin line through `fig.add_scatter`, the same linear interpolation `crossing` used. `fig.add_hline` draws `background` as a dashed line and `half_level` as a dotted line. A last `fig.add_scatter` marks `x_left` and `x_right` with crosses at the half level and joins them with a thick segment whose half-length is the half-width, given in the legend.

In [ ]:
fig = px.scatter(
    science, x='point_along_profile_m', y='residual_nettleton',
    title=f'Half-width construction on the residual at {rho_nettleton / 1000:.2f} g/cm³',
    labels={'point_along_profile_m': 'Along-profile distance (m); base at 100 m',
            'residual_nettleton': 'Residual (mGal)'},
)
fig.update_traces(marker=dict(size=12), name='stations', showlegend=True)
fig.add_scatter(x=x, y=a, mode='lines', name='linear interpolation between stations',
                line=dict(color=px.colors.qualitative.Safe[0], width=1))
fig.add_hline(y=background, line_dash='dash', annotation_text='background (median)')
fig.add_hline(y=half_level, line_dash='dot', annotation_text='half level')
fig.add_scatter(x=[x_left, x_right], y=[half_level, half_level], mode='markers+lines',
                name=f'half-level crossings, x½ = {half_width:.1f} m',
                marker=dict(size=14, symbol='x', color=px.colors.qualitative.Safe[3]),
                line=dict(color=px.colors.qualitative.Safe[3], width=3))
fig.show()

**Figure description:** The residual at the sweep density (mGal, y-axis) against along-profile distance (metres, x-axis), stations as markers joined by a thin line, with a dashed horizontal line at the background level, a dotted horizontal line at the half level, and two cross markers joined by a thick segment where the profile crosses the half level either side of the minimum. Half the length of that segment is the half-width. Non-visual path: the previous cell prints the background, half level, both crossing positions and the half-width.

**Question 7.1.** The printout gives the half-width and the two depth estimates.

1. Which simple shape, sphere or cylinder, is the better model for a sinkhole, and what about a sinkhole's geometry decides it?
2. The profile is a single line. What one piece of information, which this line cannot supply, would settle whether the sphere or cylinder rule applies? How would you collect it?

*(Your answer):*

**Question 7.2.** The stations are 10 m apart and the half-width is smaller than that spacing, so each half-level crossing is one linear interpolation between two adjacent stations.

1. What does that do to the reliability of the half-width and of the depth that follows from it?
2. If you could re-survey the 40 m around the minimum, what station spacing would you specify, and what would you expect it to do to the half-width?

*(Your answer):*

**Question 7.3.** The excess mass is printed for the sphere depth. For a spherical cavity of radius $r$ with density contrast $\Delta\rho$, the excess mass is $M_e = \tfrac{4}{3}\pi r^3\,\Delta\rho$. Take the surrounding limestone at 2.3 g/cm³ (2300 kg/m³).

1. If the cavity is air-filled, the contrast is −2300 kg/m³. What radius does the printed excess mass imply?
2. If it is water-filled, the contrast is −1300 kg/m³. What radius now?
3. Both cavities produce this same profile. What would you need to know about the site to choose between them, and could a second gravity line supply it?

*(Your answer):*

## Part 8: Reflection: two choices decided the answer

**Question 8.1.** The last two lines of the Part 7 printout compare the anomaly amplitude with the pooled repeatability of the base reads, at both slab densities.

1. At the sweep density, how many times the noise floor is the anomaly? Using the [68-95-99.7 rule](https://github.com/soule-geophysics/geol-333-714/blob/main/GLOSSARY.md#68-95-997-rule), is a single station wandering that far by measurement scatter a plausible explanation for the low?
2. From the Part 6 table, how many stations carry the low at the sweep density? What does that do to your confidence?
3. At 2.67 g/cm³ the anomaly is far larger relative to the noise. Why does a larger ratio there make that anomaly no more likely to be the sinkhole?
4. Name one additional field measurement that would most raise your confidence in the low, and say why.

*(Your answer):*

**Question 8.2.** Two reductions of the same readings put the sinkhole at two different stations, and the only difference between them was the slab density. Nettleton's criterion chose the density from the data.

1. Part 6 stated the assumption the criterion rests on: buried geology does not follow the shape of the ground surface. Describe one geological situation in which that assumption fails, and say which way it would push the chosen density.
2. Write the one-sentence conclusion you would put in a report: where the sinkhole is, what density you used and why, and what alternative you rejected.

*(Your answer):*

**Question 8.3.** This notebook applies no latitude correction. Week 4 placed one between drift and free air, because normal gravity rises toward the pole (Burger §6.3.2). The northward gradient of Eq. 6.12 is `0.811 sin 2φ` mGal per kilometre, which at the site's latitude of 28.06° is 0.673 mGal/km. The stations span 169 m of northing (`northing_m` in the survey table), and the along-profile coordinate runs south.

1. Multiply the gradient by the northing span to get the size of the latitude trend across the line, in mGal. Compare it with the change in the regional across the line, printed by the reduction above and reported in HW2a's Question 4.1, and with the anomaly amplitude from Part 7.
2. The stations lie on a straight line, so northing changes by a fixed amount per metre along the profile. What shape does the latitude trend have when plotted against `point_along_profile_m`?
3. Suppose the latitude correction had been applied before the regional was fitted. What would the fitted regional line have done with it? Would the residual, the sweep density and the position of the sinkhole have changed? Show your reasoning.

*(Your answer):*

**Question 8.4.** *(Optional, ungraded)* Which result in this dataset did you least expect, and what would you check to test it?

*(Your answer):*

## How to submit

1. Run all cells from the top. (`Runtime → Run all` in Colab.)
2. Make sure all your short answers are filled in.
3. `File → Download → Download .ipynb`.
4. Rename the file to `HW2b_LASTNAME.ipynb` (e.g., `HW2b_Smith.ipynb`).
5. Upload to the Brightspace HW2b dropbox by **Sunday October 11, 11:59 PM**.

If something is broken or unclear, post on the **Ask the Class (General Q&A)** discussion topic. Other students may have the same question.